# Train on Google Colab (free GPU)

End-to-end: clone the repo → download a dataset from Kaggle → extract face crops → train → evaluate → download the trained model to your laptop.

**Before you start**
1. Runtime → Change runtime type → **T4 GPU** → Save.
2. A free Kaggle account. Go to kaggle.com → your avatar → **Settings** → *API* → **Create New Token**. This downloads `kaggle.json`; you'll upload it below.
3. If you choose the DFDC sample, open https://www.kaggle.com/c/deepfake-detection-challenge/rules once and click **I Understand and Accept** (otherwise the download returns 403).

Typical wall-clock on a T4: download 5–10 min · face extraction 10–20 min · training 10–15 min.

Run the cells top to bottom.

In [ ]:
!nvidia-smi -L || echo "No GPU - go to Runtime > Change runtime type and pick T4 GPU" 

## 1. Get the code

In [ ]:
import os
REPO = "https://github.com/Ishan700/Explainable-Deepfake-Detection.git"
if not os.path.exists("Explainable-Deepfake-Detection"):
    !git clone -q $REPO
%cd /content/Explainable-Deepfake-Detection
# Colab already ships TensorFlow + OpenCV; only the light extras are needed.
!pip install -q scikit-learn pandas matplotlib tqdm flask kaggle
import tensorflow as tf, cv2
print("TensorFlow", tf.__version__, "| OpenCV", cv2.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

## 2. Kaggle credentials
Upload the `kaggle.json` you created in Kaggle settings when the file picker appears.

In [ ]:
import os, pathlib
from google.colab import files

kag = pathlib.Path("~/.kaggle/kaggle.json").expanduser()
if not kag.exists():
    print("Choose your kaggle.json ...")
    up = files.upload()
    kag.parent.mkdir(exist_ok=True)
    kag.write_bytes(up["kaggle.json"])
    kag.chmod(0o600)
print("Kaggle credentials ready")

## 3. Download a dataset and sort it into `data_raw/real` and `data_raw/fake`

| Choice | What you get | Download size |
|---|---|---|
| `celebdf` | Celeb-DF v2 (890 real, 5,639 fake, high-quality swaps) | ~10 GB |
| `dfdc` | DFDC Kaggle sample (400 videos, mixed quality) | ~4 GB |

`VIDEOS_PER_CLASS` caps each class so the set is balanced and extraction stays fast. 300 per class → ~9,600 face crops at 16 frames/video, which is plenty for a minor project.

In [ ]:
DATASET = "celebdf"        #@param ["celebdf", "dfdc"]
VIDEOS_PER_CLASS = 300     #@param {type:"integer"}

import shutil, os
os.makedirs("/content/raw", exist_ok=True)
if DATASET == "celebdf":
    !kaggle datasets download -d reubensuju/celeb-df-v2 -p /content/raw --unzip -q
    !python scripts/import_dataset.py --source celebdf /content/raw --limit $VIDEOS_PER_CLASS --move
else:
    !kaggle competitions download -c deepfake-detection-challenge -f train_sample_videos.zip -p /content/raw -q
    !unzip -q -o /content/raw/train_sample_videos.zip -d /content/raw
    !python scripts/import_dataset.py --source dfdc /content/raw --limit $VIDEOS_PER_CLASS --move

!echo real: $(ls data_raw/real | wc -l)   fake: $(ls data_raw/fake | wc -l)

## 4. Extract face crops (split by video, manifest written)

In [ ]:
!python prepare_dataset.py --frames 16

## 5. Train
Phase 1 trains the head on a frozen ImageNet EfficientNet-B0; phase 2 fine-tunes the top layers. Best checkpoint by validation AUC is kept.

In [ ]:
!python train.py --epochs 12 --fine-tune-epochs 6 --batch-size 32

In [ ]:
from IPython.display import Image, display
display(Image("results/training_curves.png", width=900))

## 6. Evaluate on the held-out test videos

In [ ]:
!python evaluate.py --gradcam-samples 6

In [ ]:
import json, glob
from IPython.display import Image, display
rep = json.load(open("results/eval_test.json"))
for level in ("frame_level", "video_level"):
    if level in rep:
        m = rep[level]
        print(f"{level:<12} n={m['n']:<5} acc={m['accuracy']:.3f}  auc={m.get('auc', float('nan')):.3f}  "
              f"F1(fake)={m['f1_fake']:.3f}")
for p in ["results/roc_test.png", "results/confusion_test_frames.png", "results/confusion_test_videos.png"]:
    if glob.glob(p):
        display(Image(p, width=380))
for p in sorted(glob.glob("results/gradcam_samples_test/*.jpg"))[:4]:
    display(Image(p, width=520))

## 7. Try it on one video

In [ ]:
import glob
sample = sorted(glob.glob("data_raw/fake/*"))[0]
!python predict.py "$sample" --out results/demo_prediction
from IPython.display import Image, display
display(Image("results/demo_prediction/gradcam_regions.jpg", width=360))

## 8. Download the trained model

Unzip on your laptop and copy **`model/deepfake_model.keras`** and **`model/class_names.json`** into the project's `model/` folder. Then `python app.py` gives real verdicts. The `results/` folder has the numbers and plots for your README / report.

In [ ]:
!zip -q -r trained_model.zip model/deepfake_model.keras model/class_names.json results
from google.colab import files
files.download("trained_model.zip")

### Optional: also keep a copy in Google Drive
(Colab wipes `/content` when the runtime disconnects.)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp trained_model.zip /content/drive/MyDrive/
print("Saved to Google Drive: MyDrive/trained_model.zip")